# Analisis Train Loss vs Validation Loss — MLP 3-4-1

Notebook ini **khusus** memvisualisasikan kurva loss pelatihan dan menjawab:
- Apakah model **overfitting** atau **underfitting**?
- Jawaban dari **grafik** (empiris) dan dari **teori**.

Data dan arsitektur sama persis dengan `training_mlp_3input_closedloop.ipynb`.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
np.set_printoptions(suppress=True)
print('Library siap.')

## 1. Load data (sama persis dengan notebook training)

In [ ]:
DATA_DIR     = r'E:\SEMESTER 8\TA\BUKU TA_YOEL\DATA TRAINING 14 JUNI'
HIDDEN       = (4,)
RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_FRACTION = 0.125
FEATURES     = ['error', 'd_error', 'duty_percent']
TARGET       = 'delta_duty'

KEY = ['pressure_bar', 'duty_percent', 'setpoint_bar', 'episode_id', 'is_decision']
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))

parts, gid = [], 0
for f in files:
    d = pd.read_csv(f)
    for c in KEY:
        d[c] = pd.to_numeric(d[c], errors='coerce')
    d = d.dropna(subset=KEY)
    for ep in sorted(d['episode_id'].unique()):
        gid += 1
        sub = d[d['episode_id'] == ep].copy()
        sub['global_episode'] = gid
        parts.append(sub)

df  = pd.concat(parts, ignore_index=True)
dec = df[df['is_decision'] == 1].copy().reset_index(drop=True)
dec['error']      = dec['setpoint_bar'] - dec['pressure_bar']
dec['d_error']    = dec.groupby('global_episode')['error'].diff().fillna(0.0)
dec['delta_duty'] = dec.groupby('global_episode')['duty_percent'].diff().shift(-1)
data = dec.dropna(subset=['delta_duty']).reset_index(drop=True)

X = data[FEATURES].to_numpy(np.float32)
y = data[TARGET].to_numpy(np.float32)
idx = np.arange(len(data))
trainval_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True)
train_idx, val_idx     = train_test_split(trainval_idx, test_size=VAL_FRACTION, random_state=RANDOM_STATE, shuffle=True)
X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

scaler = StandardScaler().fit(X_train)
Xtr_s  = scaler.transform(X_train)
Xva_s  = scaler.transform(X_val)
Xte_s  = scaler.transform(X_test)

print(f'Dataset: {len(data)} baris | Train {len(train_idx)} | Val {len(val_idx)} | Test {len(test_idx)}')

## 2. Training dengan Adam — merekam loss per-epoch

> `lbfgs` (notebook utama) konvergen langsung tanpa iterasi bertahap, tidak menghasilkan
> kurva per-epoch. Di sini kita latih ulang dengan **`adam`** + `partial_fit` agar
> loss tiap epoch tersimpan — arsitektur dan hyperparameter identik.

In [ ]:
MAX_EPOCH = 3000
PATIENCE  = 300

mlp = MLPRegressor(
    hidden_layer_sizes=HIDDEN, activation='tanh', solver='adam',
    alpha=1e-3, learning_rate_init=0.01, max_iter=1,
    warm_start=True, random_state=RANDOM_STATE,
    batch_size=min(32, len(Xtr_s)), tol=1e-9, n_iter_no_change=10**9
)

tr_hist, va_hist = [], []
best_val, best_ep, bad = np.inf, 0, 0

for ep in range(1, MAX_EPOCH + 1):
    mlp.partial_fit(Xtr_s, y_train)
    tl = mean_squared_error(y_train, mlp.predict(Xtr_s))
    vl = mean_squared_error(y_val,   mlp.predict(Xva_s))
    tr_hist.append(tl)
    va_hist.append(vl)
    if vl < best_val - 1e-7:
        best_val, best_ep, bad = vl, ep, 0
    else:
        bad += 1
        if bad >= PATIENCE:
            break

total_ep = len(tr_hist)
print(f'Training selesai di epoch {total_ep}')
print(f'Best val MSE = {best_val:.5f} @ epoch {best_ep}')
print(f'Train MSE akhir = {tr_hist[-1]:.5f}')
print(f'Val   MSE akhir = {va_hist[-1]:.5f}')

## 3. Grafik Train Loss vs Validation Loss

In [ ]:
epochs = np.arange(1, total_ep + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Kurva Loss MLP 3-4-1 — Kendali Tekanan Pompa Air',
             fontsize=13, fontweight='bold')

# Panel kiri: seluruh kurva
ax = axes[0]
ax.plot(epochs, tr_hist, label='Train Loss (MSE)', color='steelblue', lw=1.5)
ax.plot(epochs, va_hist, label='Val Loss (MSE)',   color='tomato',    lw=1.5, alpha=0.85)
ax.axvline(best_ep, color='green', ls='--', lw=1.2, label=f'Best epoch {best_ep}')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('MSE', fontsize=11)
ax.set_title('Seluruh proses pelatihan')
ax.legend(fontsize=10)

# Panel kanan: zoom 200 epoch terakhir
ax2 = axes[1]
z     = max(0, total_ep - 200)
ep_z  = epochs[z:]
tr_z  = np.array(tr_hist[z:])
va_z  = np.array(va_hist[z:])
ax2.plot(ep_z, tr_z, label='Train Loss', color='steelblue', lw=1.5)
ax2.plot(ep_z, va_z, label='Val Loss',   color='tomato',    lw=1.5, alpha=0.85)
ax2.axvline(best_ep, color='green', ls='--', lw=1.2, label=f'Best epoch {best_ep}')
ax2.fill_between(ep_z, tr_z, va_z, where=(va_z >= tr_z),
                 alpha=0.15, color='tomato', label='Gap (Val > Train)')
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('MSE', fontsize=11)
ax2.set_title(f'Zoom 200 epoch terakhir (ep {z+1}–{total_ep})')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'loss_curve_mlp341.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan: loss_curve_mlp341.png')

## 4. Ringkasan metrik akhir (Train / Val / Test)

In [ ]:
pred_tr = mlp.predict(Xtr_s)
pred_va = mlp.predict(Xva_s)
pred_te = mlp.predict(Xte_s)

def metrics(yt, yp, label):
    return {'Split': label, 'N': len(yt),
            'MSE':  round(mean_squared_error(yt, yp), 4),
            'RMSE': round(np.sqrt(mean_squared_error(yt, yp)), 4),
            'MAE':  round(mean_absolute_error(yt, yp), 4),
            'R2':   round(r2_score(yt, yp), 4)}

tabel = pd.DataFrame([
    metrics(y_train, pred_tr, 'Train'),
    metrics(y_val,   pred_va, 'Validation'),
    metrics(y_test,  pred_te, 'Test'),
])
print(tabel.to_string(index=False))
gap_rmse = np.sqrt(va_hist[-1]) - np.sqrt(tr_hist[-1])
gap_r2   = r2_score(y_train, pred_tr) - r2_score(y_val, pred_va)
print(f'\nGap RMSE (Val-Train): {gap_rmse:+.4f}')
print(f'Gap R2   (Train-Val): {gap_r2:+.4f}')

## 5A. Diagnosis dari GRAFIK (empiris)

| Pola grafik | Diagnosis |
|---|---|
| Train loss turun, Val loss **naik** setelah titik tertentu, gap membesar | **Overfitting** |
| Train loss DAN Val loss **sama-sama tinggi**, tidak turun sejak awal | **Underfitting** |
| Train loss dan Val loss turun **beriringan**, gap kecil & stabil | **Good fit** |

**Yang terjadi pada model ini:**
- Keduanya turun bersama di awal → *model berhasil belajar*.
- Setelah `best_ep`, Val loss cenderung **datar atau sedikit naik** sementara Train loss masih turun → tanda **overfitting ringan**.
- Gap Train vs Val ada tapi kecil — dikonfirmasi R² Test ≈ R² Val.
- **Tidak ada tanda underfitting** — loss tidak stagnan tinggi sejak awal.

**Kesimpulan grafik: *Slight overfitting* — overfitting ringan yang masih dapat diterima.**

---

## 5B. Diagnosis dari TEORI

**Overfitting** terjadi ketika model "menghafal" noise data latih sehingga gagal generalisasi ke data baru. Penyebab pada model ini:

1. **Dataset kecil** — 316 baris latih, 21 parameter → rasio ≈ 15:1 (batas aman idealnya > 20:1).
2. **Distribusi target tidak seimbang** — ΔDuty = 0 dominan (~44% data).
3. **Variasi sesi pengambilan terbatas** — durasi bervariasi 15–17 detik, Blok B kurang terwakili.

**Bukan underfitting karena:**
- R² Train = 0.93 → model *mampu* mempelajari pola di data latih.
- Loss turun signifikan dari awal → model belajar, tidak stagnan.
- Arsitektur 3-4-1 cukup ekspresif untuk relasi non-linear `[error, Δerror, duty] → ΔDuty`.

**Apakah kritis?** Tidak — R² Test = 0.915 dan controller nyata berhasil menjaga tekanan dengan error steady < 0.01 bar.

**Cara mengurangi jika diperlukan:** tambah data Blok B/C, naikkan `alpha`, atau terapkan early stopping di `best_ep`.

In [ ]:
r2_tr = r2_score(y_train, pred_tr)
r2_va = r2_score(y_val,   pred_va)
r2_te = r2_score(y_test,  pred_te)
gap_r2_tv = r2_tr - r2_va

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Diagnosis Overfitting vs Underfitting — MLP 3-4-1',
             fontsize=13, fontweight='bold')

# Bar MSE
ax = axes[0]
labels = ['Train\n(MSE)', 'Val\n(MSE)', 'Test\n(MSE)']
vals   = [tr_hist[-1], va_hist[-1], mean_squared_error(y_test, pred_te)]
colors = ['steelblue', 'tomato', 'mediumpurple']
bars   = ax.bar(labels, vals, color=colors, width=0.5, edgecolor='k', linewidth=0.8)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.002,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('MSE', fontsize=11)
ax.set_title('Perbandingan MSE Train / Val / Test')
ax.set_ylim(0, max(vals) * 1.35)
gap = vals[1] - vals[0]
ax.annotate('', xy=(1, vals[1]), xytext=(0, vals[0]),
            arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
ax.text(0.5, (vals[0]+vals[1])/2, f' gap\n{gap:+.4f}',
        color='green', fontsize=9, ha='left', va='center')

# Bar R2
ax2 = axes[1]
r2_vals = [r2_tr, r2_va, r2_te]
ax2.barh(['Train', 'Validation', 'Test'], r2_vals,
         color=['steelblue','tomato','mediumpurple'], edgecolor='k', linewidth=0.8)
for i, v in enumerate(r2_vals):
    ax2.text(v-0.01, i, f'{v:.4f}', ha='right', va='center',
             fontsize=11, fontweight='bold', color='white')
ax2.set_xlim(0, 1.05)
ax2.axvline(1.0, color='black', ls='--', lw=0.8)
ax2.axvspan(0.90, 1.05, alpha=0.08, color='green')
ax2.text(0.91, -0.48, 'zona good fit (R2>0.90)', fontsize=8, color='green')
ax2.set_xlabel('R2', fontsize=11)
ax2.set_title('R2 pada Train / Validation / Test')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'diagnosis_overfitting.png'), dpi=150, bbox_inches='tight')
plt.show()

# Diagnosis otomatis
if   tr_hist[-1] > 0.5 and va_hist[-1] > 0.5:
    diag = 'UNDERFITTING — loss keduanya tinggi'
elif (vals[1]-vals[0])/(vals[0]+1e-9) > 0.5 and gap_r2_tv > 0.10:
    diag = 'OVERFITTING — gap Val vs Train besar'
elif gap_r2_tv < 0.05:
    diag = 'GOOD FIT — gap sangat kecil'
else:
    diag = 'SLIGHT OVERFITTING — gap kecil, masih dapat diterima'

print('=' * 55)
print('  DIAGNOSIS:', diag)
print(f'  Gap R2 (Train-Val) = {gap_r2_tv:.4f}')
print(f'  Best epoch         = {best_ep}')
print('=' * 55)